# 面试问题：MHA、MQA、GQA 有什么区别？Query head 怎样映射 KV head，KV Cache 能省多少？

**一句话回答**：MHA 为每个 query head 配独立 K/V head；MQA 让所有 query heads 共享一组 K/V；GQA 把 query heads 分组，每组共享一个 KV head。计算时 `kv_head=floor(q_head/(Hq/Hkv))`，因此要求 `Hq%Hkv=0`。减少 KV heads 主要降低 decode 阶段 KV Cache 容量和读取带宽，以潜在质量损失换吞吐。

本 Notebook 用 NumPy 从投影、head 映射和 causal attention 写到增量 KV Cache、显存公式、MHA checkpoint 平均池化 uptraining 与 TP 分片；代码注释说明张量轴，并把质量、容量与读取带宽放在同一条评测曲线上。


In [ ]:
import math
import numpy as np

# 小尺寸张量用于精确比较 grouped 与显式展开实现。
SEED142=14201; rng142=np.random.default_rng(SEED142)
assert SEED142==14201
assert 8%2==0
assert np.isfinite(rng142.normal())


## 1. 先写清 `Hq/Hkv/D` 三个轴

Query 形状可写 `[B,T,Hq,D]`，Key/Value 为 `[B,T,Hkv,D]`。MHA 有 `Hkv=Hq`，MQA 有 `Hkv=1`，GQA 位于两者之间。hidden size、head dim、RoPE rotary dim 和 TP 都需满足各自整除关系。


In [ ]:
def validate_heads142(hidden,hq,hkv):
    # hidden 决定 query head dim；每组 query 数必须为整数。
    if hidden%hq or hq%hkv: raise ValueError("head_contract")
    return {"head_dim":hidden//hq,"group_size":hq//hkv}
cfg142=validate_heads142(32,8,2)
assert cfg142=={"head_dim":4,"group_size":4}
try: validate_heads142(30,8,3); raise AssertionError("bad heads")
except ValueError as e: assert str(e)=="head_contract"
assert validate_heads142(32,8,1)["group_size"]==8


## 2. Q 投影输出 Hq 个头，K/V 只输出 Hkv 个头

对 hidden 输入 `X`，`Wq` 输出 `Hq·D`，`Wk/Wv` 输出 `Hkv·D`。因此 GQA 不只是 runtime 把 K/V 复制少几份，checkpoint 参数形状本身也不同。下面显式 reshape，不调用 attention 层。


In [ ]:
T142,HIDDEN142,HQ142,HKV142,D142=5,32,8,2,4; X142=rng142.normal(size=(T142,HIDDEN142))
Wq142=rng142.normal(size=(HIDDEN142,HQ142*D142)); Wk142=rng142.normal(size=(HIDDEN142,HKV142*D142)); Wv142=rng142.normal(size=(HIDDEN142,HKV142*D142))
# reshape 后保留 token、head、head_dim 三个语义轴。
Q142=(X142@Wq142).reshape(T142,HQ142,D142); K142=(X142@Wk142).reshape(T142,HKV142,D142); V142=(X142@Wv142).reshape(T142,HKV142,D142)
assert Q142.shape==(5,8,4)
assert K142.shape==V142.shape==(5,2,4)
assert Wk142.shape[1]==Wq142.shape[1]//4


## 3. Query heads 按连续组映射到 KV heads

group size 为 `G=Hq/Hkv`，第 h 个 query head 使用 `h//G` 的 K/V。生产 kernel 可避免物理 repeat，只在索引时映射；教学实现先显式展开建立 oracle。checkpoint 必须记录 head 排列，不能假设所有模型都采用相同 interleave。


In [ ]:
def kv_map142(hq,hkv):
    # 连续 query heads 共享同一个 KV head。
    g=hq//hkv; return np.arange(hq)//g
mapping142=kv_map142(8,2); Kexp142=K142[:,mapping142,:]
assert np.array_equal(mapping142,[0,0,0,0,1,1,1,1])
assert Kexp142.shape==Q142.shape
assert np.array_equal(Kexp142[:,0],Kexp142[:,3])


## 4. 共享 K/V 不改变每个 query head 的 softmax 语义

每个 q head 仍独立计算 score、causal mask、softmax 和 value 聚合，只是读取的 K/V head 相同。下面直接按映射循环，并与物理展开 K/V 的通用实现比较，验证数值等价。


In [ ]:
def stable_softmax142(x):
    x=x-np.max(x,axis=-1,keepdims=True); e=np.exp(x); return e/e.sum(axis=-1,keepdims=True)
def gqa142(q,k,v):
    # 输出仍有 Hq 个独立 head；仅 K/V 索引被分组共享。
    T,hq,d=q.shape; hkv=k.shape[1]; out=np.empty_like(q); mapping=kv_map142(hq,hkv)
    causal=np.arange(T)[None,:]<=np.arange(T)[:,None]
    for h,kh in enumerate(mapping):
        s=q[:,h]@k[:,kh].T/math.sqrt(d); s=np.where(causal,s,-np.inf); out[:,h]=stable_softmax142(s)@v[:,kh]
    return out
out142=gqa142(Q142,K142,V142); expanded142=gqa142(Q142,K142[:,mapping142],V142[:,mapping142])
assert np.allclose(out142,expanded142)
assert out142.shape==Q142.shape
assert np.all(np.isfinite(out142))


## 5. KV Cache 容量与 Hkv 线性相关

每层大约保存 `2·B·T·Hkv·D·bytes`，再乘层数。MHA→GQA 的 query 参数和 attention FLOPs 主体仍在，但历史 K/V 容量与 decode 读取带宽按 `Hkv/Hq` 缩小。实际还包含 block padding、量化 scale 和 allocator 元数据。


In [ ]:
def kv_bytes142(batch,tokens,layers,hkv,d,bytes_=2):
    # 前面的 2 分别代表 Key 与 Value。
    return 2*batch*tokens*layers*hkv*d*bytes_
mha_mem142=kv_bytes142(4,4096,32,8,128); gqa_mem142=kv_bytes142(4,4096,32,2,128); mqa_mem142=kv_bytes142(4,4096,32,1,128)
assert mha_mem142==4*gqa_mem142
assert gqa_mem142==2*mqa_mem142
assert mqa_mem142>0


## 6. Decode 只 append 新 K/V，并让新 query 读取全部合法历史

prefill 一次生成前缀 K/V；每个 decode step 只投影当前 token 的 K/V 并追加。新 token 位于序列末端，可以读取全部 cache。用相同绝对 position/ RoPE，增量最后一步应与整段 causal forward 的最后位置一致。


In [ ]:
def decode_last142(q_last,k_cache,v_cache):
    # q_last 形状 [Hq,D]，cache 仍按 [T,Hkv,D] 存储。
    hq,d=q_last.shape; mapping=kv_map142(hq,k_cache.shape[1]); out=np.empty_like(q_last)
    for h,kh in enumerate(mapping):
        s=k_cache[:,kh]@q_last[h]/math.sqrt(d); p=np.exp(s-s.max()); p/=p.sum(); out[h]=p@v_cache[:,kh]
    return out
step142=decode_last142(Q142[-1],K142,V142)
assert np.allclose(step142,out142[-1])
assert step142.shape==(8,4)
assert np.all(np.isfinite(step142))


## 7. MHA checkpoint 可按组池化 K/V 后再短程 uptraining

将同组 MHA K/V projection heads 求均值，可初始化 GQA 权重；若组内头本来相同，转换精确。一般情况下注意力行为会变化，所以仍需少量训练恢复质量，并验证 head 排列、bias 和 optimizer state 的迁移。


In [ ]:
def pool_heads142(W,hq,hkv,d):
    # 输入权重最后一维先还原 head 轴，再对组内 query heads 平均。
    return W.reshape(W.shape[0],hkv,hq//hkv,d).mean(axis=2).reshape(W.shape[0],hkv*d)
Wmha142=np.repeat(rng142.normal(size=(32,2,4))[:, :, None, :],4,axis=2).reshape(32,32)
pooled142=pool_heads142(Wmha142,8,2,4)
assert pooled142.shape==(32,8)
assert np.allclose(pooled142,Wmha142.reshape(32,2,4,4)[:,:,0,:].reshape(32,8))
assert np.all(np.isfinite(pooled142))


## 8. Tensor Parallel 要决定 KV head 是切分还是复制

若 `Hkv≥TP` 且整除，可每 rank 切 K/V heads；若 `Hkv<TP`，常在部分 ranks 复制 KV head 或调整 TP。映射必须保证本地 query 能访问对应 K/V，且 checkpoint manifest 保存全局到本地 head ID。


In [ ]:
def tp_plan142(hq,hkv,tp):
    # 返回每 rank 的 query 数，以及 KV 采用 shard 还是 replication。
    if hq%tp: raise ValueError("q_not_divisible")
    mode="shard" if hkv>=tp and hkv%tp==0 else "replicate"
    return {"q_per_rank":hq//tp,"kv_mode":mode,"kv_per_rank":hkv//tp if mode=="shard" else 1}
assert tp_plan142(8,2,2)["kv_mode"]=="shard"
assert tp_plan142(8,1,4)["kv_mode"]=="replicate"
assert tp_plan142(8,2,4)["q_per_rank"]==2


## 面试总结

完整主线是：**明确 `[T,Hq,D]` 与 `[T,Hkv,D]` → `G=Hq/Hkv` → `q_head//G` 映射 → 每个 Q head 独立 softmax → KV 容量 `2BTLHkvDbytes` → append-only decode 与全量最后位对齐 → MHA K/V 组内池化初始化并 uptrain → TP 下 shard/replicate KV → 比较质量、TTFT/TPOT、带宽与显存**。GQA 主要优化 decode 的 KV 成本，不是免费等价替换。

延伸阅读：[GQA](https://arxiv.org/abs/2305.13245)、[Multi-Query Attention](https://arxiv.org/abs/1911.02150)、[FlashAttention](https://arxiv.org/abs/2205.14135)。
